# v8 실행 전 정리 (Cleanup)

**목적**: v6/v7의 잘못된 MapCube와 거기서 파생된 모든 srcmap, component map을 삭제하여 v8이 처음부터 깨끗하게 실행되도록 보장.

**핵심**: v6/v7의 MapCube는 14-bin으로 잘못 재샘플링되어 있어, 그대로 두면 Cell 4가 `os.path.exists()` 체크에서 건너뛸 수 있습니다. 모든 파생 산출물도 같은 이유로 삭제 필요.

**안전장치**: 기본은 dry-run (목록만 표시). 실제 삭제는 마지막 확인 셀에서 수동으로.

In [9]:
# %% ══════════════════════════════════════════════════════════
# Cleanup Cell 1: 삭제 대상 스캔 (DRY RUN — 아무것도 삭제 안 함)
# ══════════════════════════════════════════════════════════════

import os
import shutil
from pathlib import Path

# ⚠️ WORK_DIR 지정 (v8 노트북과 동일해야 함)
WORK_DIR = '/home/haebarg/GCE-Chi-square-fitting/GCE_12yr_reproduce'

# 삭제할 폴더 (v6/v7에서 만들어진 모든 중간 산출물)
DIRS_TO_DELETE = [
    'MapCubes',                  # ⭐ 가장 중요 — 14-bin 잘못된 cube
    'Source_Maps',               # convolved srcmap (잘못된 MapCube 기반)
    'Source_Maps_NoConvol',      # non-convol srcmap
    'Component_Maps',            # convolved gtmodel output
    'Component_Maps_NoConvol',   # non-convol gtmodel output
]

# 삭제할 개별 파일 (v6/v7 결과 파일들)
FILES_TO_DELETE = [
    'GCE_template_NFW2.fits',          # 이전 NFW² template (v8이 다시 만듦)
    'Fermi_Bubbles_template.fits',     # 이전 Bubbles template
    'GCE_results_v5.pkl',              # 이전 결과들
    'GCE_results_v6.pkl',
    'GCE_results_v7.pkl',
    'GCE_fit_summary_v5.csv',
    'GCE_fit_summary_v6.csv',
    'GCE_fit_summary_v7.csv',
    'cov_sys_pipeline_v6.npy',
    'cov_total_pipeline_v6.npy',
    'flux_matrix_80models_v6.npy',
    'cov_sys_pipeline_v7.npy',
    'cov_total_pipeline_v7.npy',
    'flux_matrix_80models_v7.npy',
    'bubble_constraints.txt',          # v8이 다시 만듦
    'iso_constraints_full_err.txt',    # v8이 다시 만듦
]

# ⚠️ 보존 (절대 삭제 안 함)
PROTECTED_FILES = [
    'Allsky_select_12yr_front_clean.fits',  # raw filtered events
    'Allsky_gti_12yr_front_clean.fits',     # gtmktime output
    'Allsky_ltcube_12yr_front_clean.fits',  # ltcube (시간 오래 걸림)
    'GC_ccube_12yr_front_clean.fits',       # CCUBE (gtbin output)
    'GC_expcube_center_12yr_front_clean.fits',  # exposure cube center
    'Allsky_expcube_edge_12yr_front_clean.fits', # exposure cube edge
    'bin_definitions.fits',                 # gtbindef output
    'sr_per_pixel.npy',                     # cached steradian map
    'disk_mask.npy',                        # disk mask
    'fermi_bubbles.txt',                    # bubble vertices (사용자 디지타이즈)
    # XML 파일들
    'GC_psc_model.xml',
    'GC_psc_model_DR2.xml',
    'empty_model.xml',
]

print('=' * 70)
print('v8 Cleanup — DRY RUN (실제 삭제 안 함)')
print('=' * 70)
print(f'WORK_DIR: {WORK_DIR}')
print()

if not os.path.exists(WORK_DIR):
    print(f'❌ WORK_DIR가 존재하지 않습니다: {WORK_DIR}')
    print('   v8 노트북의 WORK_DIR과 일치하는지 확인하세요.')
else:
    # 1. 삭제할 폴더 스캔
    total_dir_size = 0
    total_dir_count = 0
    print('[A] 삭제 대상 폴더:')
    print('-' * 70)
    for d in DIRS_TO_DELETE:
        full = os.path.join(WORK_DIR, d)
        if os.path.isdir(full):
            n_files = sum(1 for _ in Path(full).rglob('*') if _.is_file())
            size = sum(f.stat().st_size for f in Path(full).rglob('*') if f.is_file())
            total_dir_size += size
            total_dir_count += n_files
            print(f'  {d:30s} {n_files:5d} files  {size/1e9:.2f} GB')
        else:
            print(f'  {d:30s} (없음, skip)')
    print(f'  {"─"*30} {"─"*5}        {"─"*7}')
    print(f'  {"합계":30s} {total_dir_count:5d} files  {total_dir_size/1e9:.2f} GB')

    # 2. 삭제할 개별 파일 스캔
    print()
    print('[B] 삭제 대상 개별 파일:')
    print('-' * 70)
    file_total_size = 0
    file_total_count = 0
    for f in FILES_TO_DELETE:
        full = os.path.join(WORK_DIR, f)
        if os.path.isfile(full):
            size = os.path.getsize(full)
            file_total_size += size
            file_total_count += 1
            print(f'  {f:40s} {size/1e6:8.2f} MB')
        else:
            print(f'  {f:40s} (없음, skip)')
    print(f'  {"─"*40} {"─"*8}')
    print(f'  {"합계":40s} {file_total_size/1e9:8.2f} GB ({file_total_count}개)')

    # 3. 보존되는 파일 확인
    print()
    print('[C] 보존되는 파일 (재생성하면 시간 오래 걸리는 것들):')
    print('-' * 70)
    for f in PROTECTED_FILES:
        full = os.path.join(WORK_DIR, f)
        if os.path.exists(full):
            size = os.path.getsize(full) if os.path.isfile(full) else 0
            mark = '✅'
            print(f'  {mark} {f:50s} {size/1e6:8.2f} MB')
        else:
            print(f'  ⚠️  {f:50s} (없음 — 처음부터 다시 만들어야 함)')

    # 총합
    grand_total = total_dir_size + file_total_size
    print()
    print('=' * 70)
    print(f'총 삭제 예정: {grand_total/1e9:.2f} GB '
          f'({total_dir_count + file_total_count} items)')
    print('=' * 70)
    print()
    print('⚠️  실제 삭제는 다음 셀(Cleanup Cell 2)에서 수동 confirm 후 실행')
    print('   위 목록을 확인하고 보존 파일이 모두 ✅인지 체크하세요')

v8 Cleanup — DRY RUN (실제 삭제 안 함)
WORK_DIR: /home/haebarg/GCE-Chi-square-fitting/GCE_12yr_reproduce

[A] 삭제 대상 폴더:
----------------------------------------------------------------------
  MapCubes                         241 files  13.19 GB
  Source_Maps                       69 files  6.15 GB
  Source_Maps_NoConvol              69 files  8.48 GB
  Component_Maps                     0 files  0.00 GB
  Component_Maps_NoConvol            0 files  0.00 GB
  ────────────────────────────── ─────        ───────
  합계                               379 files  27.81 GB

[B] 삭제 대상 개별 파일:
----------------------------------------------------------------------
  GCE_template_NFW2.fits                       1.44 MB
  Fermi_Bubbles_template.fits                  1.44 MB
  GCE_results_v5.pkl                       (없음, skip)
  GCE_results_v6.pkl                       (없음, skip)
  GCE_results_v7.pkl                       (없음, skip)
  GCE_fit_summary_v5.csv                   (없음, skip)
  GCE_fit_summary_v6

In [10]:
# %% ══════════════════════════════════════════════════════════
# Cleanup Cell 2: 백업 권장 (선택사항, 실행 권장)
# ══════════════════════════════════════════════════════════════
# 만에 하나를 위해 v7 결과만 별도 위치로 백업

import shutil
from datetime import datetime

BACKUP_DIR = os.path.join(WORK_DIR, f'_backup_v7_{datetime.now().strftime("%Y%m%d_%H%M%S")}')
BACKUP_FILES = [
    'GCE_results_v7.pkl',
    'GCE_fit_summary_v7.csv',
    'cov_sys_pipeline_v7.npy',
    'cov_total_pipeline_v7.npy',
    'flux_matrix_80models_v7.npy',
    'bubble_constraints.txt',
    'iso_constraints_full_err.txt',
]

do_backup = True  # 백업 안 하려면 False로

if do_backup:
    os.makedirs(BACKUP_DIR, exist_ok=True)
    backed_up = 0
    for f in BACKUP_FILES:
        src = os.path.join(WORK_DIR, f)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(BACKUP_DIR, f))
            backed_up += 1
            print(f'  📦 백업: {f}')
    if backed_up == 0:
        # 빈 백업 폴더 정리
        os.rmdir(BACKUP_DIR)
        print('백업할 v7 파일 없음')
    else:
        print(f'\n✅ {backed_up}개 파일을 {BACKUP_DIR}에 백업했습니다')
        print('   (총 용량 작음 — pkl + csv + npy만)')
else:
    print('백업 비활성화됨')

백업할 v7 파일 없음


In [11]:
# %% ══════════════════════════════════════════════════════════
# Cleanup Cell 3: ⚠️ 실제 삭제 (수동 confirm 필요)
# ══════════════════════════════════════════════════════════════
# 위 Cell 1의 dry-run 출력을 검토한 후, 정말 삭제하려면
# 아래 CONFIRM_DELETE 변수를 True로 바꾸고 다시 실행

CONFIRM_DELETE = True # ← 이걸 True로 바꿔야 실제 삭제됨

if not CONFIRM_DELETE:
    print('⛔ CONFIRM_DELETE = False')
    print('   삭제를 진행하려면 위 변수를 True로 변경하고 셀을 다시 실행하세요')
    print('   먼저 Cleanup Cell 1의 dry-run 출력을 검토하세요')
else:
    print('=' * 70)
    print('🗑️  v6/v7 산출물 삭제 진행')
    print('=' * 70)

    # 폴더 삭제
    deleted_dirs = 0
    for d in DIRS_TO_DELETE:
        full = os.path.join(WORK_DIR, d)
        if os.path.isdir(full):
            try:
                shutil.rmtree(full)
                print(f'  ✅ 삭제됨: {d}/')
                deleted_dirs += 1
            except Exception as e:
                print(f'  ❌ 실패: {d}/ — {e}')

    # 개별 파일 삭제
    deleted_files = 0
    for f in FILES_TO_DELETE:
        full = os.path.join(WORK_DIR, f)
        if os.path.isfile(full):
            try:
                os.remove(full)
                print(f'  ✅ 삭제됨: {f}')
                deleted_files += 1
            except Exception as e:
                print(f'  ❌ 실패: {f} — {e}')

    print()
    print(f'완료: {deleted_dirs}개 폴더 + {deleted_files}개 파일 삭제')

    # 사후 검증: 보존 파일들이 그대로 있는지
    print()
    print('보존 파일 검증:')
    all_ok = True
    for f in PROTECTED_FILES:
        full = os.path.join(WORK_DIR, f)
        if os.path.exists(full):
            print(f'  ✅ {f}')
        else:
            print(f'  ⚠️  {f} 없음 (원래도 없었을 수 있음)')
    print()
    print('🚀 이제 v8 노트북을 처음부터 실행할 수 있습니다')

🗑️  v6/v7 산출물 삭제 진행
  ✅ 삭제됨: MapCubes/
  ✅ 삭제됨: Source_Maps/
  ✅ 삭제됨: Source_Maps_NoConvol/
  ✅ 삭제됨: Component_Maps/
  ✅ 삭제됨: Component_Maps_NoConvol/
  ✅ 삭제됨: GCE_template_NFW2.fits
  ✅ 삭제됨: Fermi_Bubbles_template.fits

완료: 5개 폴더 + 2개 파일 삭제

보존 파일 검증:
  ⚠️  Allsky_select_12yr_front_clean.fits 없음 (원래도 없었을 수 있음)
  ⚠️  Allsky_gti_12yr_front_clean.fits 없음 (원래도 없었을 수 있음)
  ⚠️  Allsky_ltcube_12yr_front_clean.fits 없음 (원래도 없었을 수 있음)
  ⚠️  GC_ccube_12yr_front_clean.fits 없음 (원래도 없었을 수 있음)
  ⚠️  GC_expcube_center_12yr_front_clean.fits 없음 (원래도 없었을 수 있음)
  ⚠️  Allsky_expcube_edge_12yr_front_clean.fits 없음 (원래도 없었을 수 있음)
  ✅ bin_definitions.fits
  ✅ sr_per_pixel.npy
  ✅ disk_mask.npy
  ✅ fermi_bubbles.txt
  ⚠️  GC_psc_model.xml 없음 (원래도 없었을 수 있음)
  ⚠️  GC_psc_model_DR2.xml 없음 (원래도 없었을 수 있음)
  ⚠️  empty_model.xml 없음 (원래도 없었을 수 있음)

🚀 이제 v8 노트북을 처음부터 실행할 수 있습니다


In [12]:
# %% ══════════════════════════════════════════════════════════
# Cleanup Cell 4: 사후 검증 (삭제 후 실행 — 깨끗한 상태인지 확인)
# ══════════════════════════════════════════════════════════════

print('=' * 70)
print('Cleanup 사후 검증')
print('=' * 70)

# 1. 삭제 대상이 정말 사라졌는지
print('\n[A] 삭제 대상 (모두 없어야 함):')
all_clean = True
for d in DIRS_TO_DELETE:
    full = os.path.join(WORK_DIR, d)
    status = '❌ 아직 있음!' if os.path.isdir(full) else '✅ 삭제됨'
    print(f'  {d:30s} {status}')
    if os.path.isdir(full):
        all_clean = False

for f in FILES_TO_DELETE:
    full = os.path.join(WORK_DIR, f)
    if os.path.isfile(full):
        print(f'  {f:40s} ❌ 아직 있음!')
        all_clean = False

# 2. 보존 파일이 그대로 있는지
print('\n[B] 보존 파일 (모두 있어야 함):')
missing_critical = []
for f in PROTECTED_FILES:
    full = os.path.join(WORK_DIR, f)
    if os.path.exists(full):
        print(f'  ✅ {f}')
    else:
        print(f'  ⚠️  {f} 없음')
        if f in ['Allsky_ltcube_12yr_front_clean.fits',
                 'GC_ccube_12yr_front_clean.fits',
                 'GC_expcube_center_12yr_front_clean.fits',
                 'Allsky_expcube_edge_12yr_front_clean.fits']:
            missing_critical.append(f)

print()
if all_clean and not missing_critical:
    print('🎯 완벽! v8을 처음부터 실행할 준비가 되었습니다')
    print()
    print('다음 단계:')
    print('  1. v8 노트북 (GCE_12yr_reproduce_v8.ipynb) 열기')
    print('  2. Cell 1 (config) → Cell 4 (MapCube) 까지만 먼저 실행')
    print('  3. MapCube 검증:')
    print('       from astropy.io import fits')
    print('       with fits.open("MapCubes/pi0_mapcube_modelX.fits") as h:')
    print('           print("shape:", h[0].data.shape)  # (38, 600, 600) 같아야 함')
    print('           print("max:", h[0].data.max())     # ~1e-6 같아야 함')
    print('  4. 정상이면 Cell 5~ 끝까지 실행')
elif missing_critical:
    print('⚠️  중요 파일이 사라졌습니다:')
    for f in missing_critical:
        print(f'   - {f}')
    print('   v8 노트북의 Section 1, 2를 다시 실행해서 재생성해야 합니다')
    print('   (gtmktime, gtltcube, gtbin, gtexpcube2 — 시간 오래 걸림)')
else:
    print('❌ 일부 파일이 아직 남아있음. Cell 3의 CONFIRM_DELETE = True로 다시 실행')

Cleanup 사후 검증

[A] 삭제 대상 (모두 없어야 함):
  MapCubes                       ✅ 삭제됨
  Source_Maps                    ✅ 삭제됨
  Source_Maps_NoConvol           ✅ 삭제됨
  Component_Maps                 ✅ 삭제됨
  Component_Maps_NoConvol        ✅ 삭제됨

[B] 보존 파일 (모두 있어야 함):
  ⚠️  Allsky_select_12yr_front_clean.fits 없음
  ⚠️  Allsky_gti_12yr_front_clean.fits 없음
  ⚠️  Allsky_ltcube_12yr_front_clean.fits 없음
  ⚠️  GC_ccube_12yr_front_clean.fits 없음
  ⚠️  GC_expcube_center_12yr_front_clean.fits 없음
  ⚠️  Allsky_expcube_edge_12yr_front_clean.fits 없음
  ✅ bin_definitions.fits
  ✅ sr_per_pixel.npy
  ✅ disk_mask.npy
  ✅ fermi_bubbles.txt
  ⚠️  GC_psc_model.xml 없음
  ⚠️  GC_psc_model_DR2.xml 없음
  ⚠️  empty_model.xml 없음

⚠️  중요 파일이 사라졌습니다:
   - Allsky_ltcube_12yr_front_clean.fits
   - GC_ccube_12yr_front_clean.fits
   - GC_expcube_center_12yr_front_clean.fits
   - Allsky_expcube_edge_12yr_front_clean.fits
   v8 노트북의 Section 1, 2를 다시 실행해서 재생성해야 합니다
   (gtmktime, gtltcube, gtbin, gtexpcube2 — 시간 오래 걸림)
